In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json
import ipywidgets as widgets
from ipywidgets import interact

In [2]:
# GARANTIR RENDERIZAÇÃO NO VS CODE
pio.renderers.default = "vscode"

In [3]:
# --- BLOCO 3: CARGA DE DADOS EXTERNOS ---
try:
    with open('fluidos.json', 'r') as f:
        fluidos = json.load(f)
    print(f"✅ {len(fluidos)} fluidos carregados do banco de dados.")
except FileNotFoundError:
    print("❌ Erro: Arquivo 'fluidos.json' não encontrado na pasta.")

✅ 13 fluidos carregados do banco de dados.


In [4]:
# MOTOR DE CÁLCULO (PENG-ROBINSON)
def calcular_pvt(fluido_nome, T_c):
    p = fluidos[fluido_nome]
    T = T_c + 273.15
    R = 8.314
    
    pressões_mpa = np.linspace(0.1, 30, 100)
    z_lista = []
    
    for P_mpa in pressões_mpa:
        P = P_mpa * 1e6
        Tr = T / p["Tc"]
        kappa = 0.37464 + 1.54226*p["omega"] - 0.26992*p["omega"]**2
        alpha = (1 + kappa * (1 - np.sqrt(Tr)))**2
        a = (0.45724 * R**2 * p["Tc"]**2 / p["Pc"]) * alpha
        b = 0.07780 * R * p["Tc"] / p["Pc"]
        
        A = (a * P) / (R**2 * T**2)
        B = (b * P) / (R * T)
        
        # Equação Cúbica: Z^3 - (1-B)Z^2 + (A-2B-3B^2)Z - (AB-B^2-B^3) = 0
        coefs = [1, -(1 - B), (A - 2*B - 3*B**2), -(A*B - B**2 - B**3)]
        raizes = np.roots(coefs)
        z_real = raizes[np.isreal(raizes)].real
        z_lista.append(np.max(z_real))

    # 3. CONSTRUÇÃO DO GRÁFICO
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=pressões_mpa, y=z_lista, name='Gás Real (PR)', line=dict(color='#0047AB', width=4)))
    fig.add_hline(y=1.0, line_dash="dash", line_color="red", annotation_text="Gás Ideal")
    
    fig.update_layout(
        title=f"Comportamento PVT: {fluido_nome} a {T_c}°C",
        xaxis_title="Pressão (MPa)", yaxis_title="Fator de Compressibilidade (Z)",
        template="simple_white", margin=dict(l=20, r=20, t=50, b=20)
    )
    fig.show()

In [5]:
# INTERFACE (ATUALIZADA) ---
print("🧪 SIMULADOR PVT - ENGENHARIA QUÍMICA (ANP)")
interact(calcular_pvt, 
         fluido_nome=widgets.Dropdown(
             options=list(fluidos.keys()), 
             description='Fluido:',
             style={'description_width': 'initial'}
         ),
         T_c=widgets.IntSlider(
             min=-100, max=400, step=5, value=25, 
             description='Temp. (°C):',
             continuous_update=False # Melhora a resposta no VS Code
         ))

🧪 SIMULADOR PVT - ENGENHARIA QUÍMICA (ANP)


interactive(children=(Dropdown(description='Fluido:', options=('metano', 'etano', 'propano', 'n-butano', 'i-bu…

<function __main__.calcular_pvt(fluido_nome, T_c)>